# m15 lab · Patient safety

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

A mapping system connects electronics to electrodes inside the heart. You compute how much mains current can leak through isolation barriers, why current through an intracardiac electrode is so much more dangerous than the same current through the skin, how a protection network handles a defibrillation pulse, and how RF ablation energy must be filtered before it reaches a nonlinear amplifier.

| Part | What you do | Equation |
| --- | --- | --- |
| A | Leakage through isolation capacitance | eq-m15-leakage |
| B | Microshock and current density | eq-m15-density |
| C | Defibrillation protection | eq-m15-defib |
| D | RF ablation: filtering and demodulation | eq-m15-rf-filter, eq-m15-demod |
| E | A safety budget | all |

SI units. The thresholds in part A are from Swerdlow et al. (1999); every other value (capacitances, voltages, resistances, pulse lengths, areas) is an illustrative choice, not a design limit or the value of any product. This lab teaches the physics; it is not a substitute for the safety standards. Each part ends with **checks**.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import safety as S

## Part A · Leakage through isolation capacitance

An isolation barrier is never perfect: its capacitance $C$ passes a current $2\pi f C V$ at mains frequency
(eq-m15-leakage). Several devices on the same patient, on the same mains, add in phase in the worst case. Compare with
the minimum 60-Hz intracardiac currents that caused intermittent capture (20 µA) and fibrillation (49 µA) in 40
patients, and the then-current 50 µA single-fault standard they discuss (Swerdlow et al. 1999).

In [ ]:
sw = S.SWERDLOW_1999_UA
for v, f in [(230.0, 50.0), (120.0, 60.0)]:
    print(f"{v:.0f} V, {f:.0f} Hz: 100 pF passes {S.capacitive_current(v, f, 100e-12) * 1e6:.2f} µA; "
          f"max capacitance for 20 µA {S.max_capacitance(20e-6, v, f) * 1e12:.0f} pF, for 50 µA {S.max_capacitance(50e-6, v, f) * 1e12:.0f} pF")
c = np.linspace(0, 1000e-12, 200)
for n in (1, 3, 5):
    plt.plot(c * 1e12, n * S.capacitive_current(230.0, 50.0, c) * 1e6, label=f"{n} device(s)")
for k, col in (("intermittent_capture", "C1"), ("ventricular_fibrillation", "C3")):
    plt.axhline(sw[k], ls="--", c=col, label=f"{k.replace('_', ' ')} ({sw[k]:.0f} µA)")
plt.xlabel("isolation capacitance per device (pF)"); plt.ylabel("leakage (µA rms)"); plt.legend(fontsize=8); plt.show()
print("three 200 pF devices, in phase:", S.total_leakage([S.capacitive_current(230, 50, 200e-12)] * 3) * 1e6, "µA")
print("two of them in antiphase:      ", S.total_leakage([S.capacitive_current(230, 50, 200e-12)] * 2, [0, 180]) * 1e6, "µA")

In [ ]:
# Checks
assert abs(S.max_capacitance(50e-6, 230.0, 50.0) - 692e-12) < 1e-12
assert sw["intermittent_capture"] < sw["us_standard_single_fault"]

## Part B · Microshock and current density

The same current is far more dangerous when it enters the heart through a small electrode. At a hemispherical electrode
of radius $a$ the current density is $I/(2\pi a^2)$; spread over a body cross-section $A$ it is about $I/A$
(eq-m15-density). Use an illustrative 1 mm radius and a 0.05 m² cross-section.

In [ ]:
a, A = 1e-3, 0.05
print(f"concentration ratio A / (2 pi a^2) = {S.density_ratio(a, A):.0f}")
r = np.linspace(a, 10e-3, 200)
plt.semilogy(r * 1e3, S.density_at(20e-6, r), label="20 µA from a 1 mm electrode")
plt.axhline(20e-6 / A, c="k", ls="--", label="20 µA spread over 0.05 m²")
plt.xlabel("distance from the electrode center (mm)"); plt.ylabel("current density (A/m²)"); plt.legend(); plt.show()

In [ ]:
# Checks
assert abs(S.density_ratio(1e-3, 0.05) - 0.05 / (2 * np.pi * 1e-6)) < 1e-6
assert abs(S.density_at(1.0, 2e-3) / S.density_at(1.0, 1e-3) - 0.25) < 1e-12

## Part C · Defibrillation protection

During external defibrillation, the recording electrodes see part of the pulse. A series resistor $R$ and a clamp at
$V_c$ protect the input: the current is $(V - V_c)/R$, the resistor absorbs $(V - V_c)^2 T/R$, and the network takes a
small fraction of the energy meant for the heart (eq-m15-defib). All values are illustrative: 2 kV across a 75 Ω body,
500 V reaching one input, a 10 V clamp, a 10 ms pulse.

In [ ]:
v_body, r_body, v_in, v_c, T = 2000.0, 75.0, 500.0, 10.0, 10e-3
for R in (1e3, 10e3, 100e3):
    print(f"R = {R/1e3:5.0f} kΩ: current {S.clamp_current(v_in, v_c, R) * 1e3:7.2f} mA, resistor energy {S.resistor_energy(v_in, v_c, R, T):7.4f} J, "
          f"share of body power {100 * S.diverted_fraction(v_in, v_c, R, v_body, r_body):.4f}%")

In [ ]:
# Checks
assert abs(S.resistor_energy(v_in, v_c, 10e3, T) * 10 / S.resistor_energy(v_in, v_c, 1e3, T) - 1) < 1e-12
assert S.clamp_current(5.0, v_c, 1e3) == 0

## Part D · RF ablation: filtering and demodulation

RF ablation runs at 300 kHz to 1 MHz (Berjano 2006). Volts of RF can reach the recording inputs, against microvolt
signals. An $n$-th-order low-pass at corner $f_c$ attenuates by about $(f_c/f)^n$ (eq-m15-rf-filter). It must act before
any nonlinearity: a weak square law $y = x + \beta x^2$ turns $A\cos\omega t$ into a baseband shift $\beta A^2/2$
(eq-m15-demod), which no later filter can remove. Illustrative: 10 V of RF at 500 kHz, a 1 µV allowed residual, a 500 Hz
corner, $\beta$ = 0.01 per volt.

In [ ]:
for fc in (200.0, 500.0, 2000.0):
    print(f"corner {fc:6.0f} Hz: order needed {S.order_needed(10.0, 1e-6, 500e3, fc)}")
fs = 20e6
t = np.arange(0, 2e-3, 1 / fs)
burst = np.where((t > 0.5e-3) & (t < 1.5e-3), 1.0, 0.0)
x = 0.1 * burst * np.cos(2 * np.pi * 500e3 * t)        # 0.1 V of RF during the burst
y = S.square_law(x, 0.01)
from scipy.signal import butter, sosfiltfilt
sos = butter(4, 20e3, fs=fs, output="sos")
plt.plot(t * 1e3, sosfiltfilt(sos, y) * 1e6, label="nonlinearity first, then low-pass")
plt.plot(t * 1e3, S.square_law(sosfiltfilt(sos, x), 0.01) * 1e6, label="low-pass first")
plt.xlabel("time (ms)"); plt.ylabel("baseband output (µV)"); plt.legend(); plt.show()
print("predicted shift beta A^2 / 2 =", S.demodulated_offset(0.1, 0.01) * 1e6, "µV")

In [ ]:
# Checks
mid = (t > 0.8e-3) & (t < 1.2e-3)
assert abs(np.mean(sosfiltfilt(sos, y)[mid]) - S.demodulated_offset(0.1, 0.01)) < 0.05 * S.demodulated_offset(0.1, 0.01)
assert np.max(np.abs(S.square_law(sosfiltfilt(sos, x), 0.01)[mid])) < 1e-3 * S.demodulated_offset(0.1, 0.01)

## Part E · A safety budget

Put the pieces in one table for an illustrative system of three devices on one patient.

In [ ]:
rows = [
    ("leakage, 3 devices x 50 pF at 230 V, 50 Hz (µA)", S.total_leakage([S.capacitive_current(230, 50, 50e-12)] * 3) * 1e6),
    ("leakage, 3 devices x 150 pF at 230 V, 50 Hz (µA)", S.total_leakage([S.capacitive_current(230, 50, 150e-12)] * 3) * 1e6),
    ("current-density ratio, 1 mm electrode vs 0.05 m²", S.density_ratio(1e-3, 0.05)),
    ("defibrillation: resistor energy, 10 kΩ (J)", S.resistor_energy(500, 10, 10e3, 10e-3)),
    ("RF: Butterworth order for 10 V to 1 µV at 500 kHz, fc 500 Hz", S.order_needed(10, 1e-6, 500e3, 500)),
]
for name, value in rows:
    print(f"{name:62s} {value:10.3f}")

In [ ]:
# Checks
assert rows[0][1] < S.SWERDLOW_1999_UA["intermittent_capture"] < rows[1][1]

## Where to go next

- Add a phase difference between devices on different mains phases and see the total fall.
- Replace the square law with a diode model and compare the demodulated shift.
- m28 applies the same protection questions to pulsed field ablation.